# 01. Daily REST API Extraction Task (Production Grade)

Extracts previous day's e-commerce records via REST API or enterprise simulation engine:
- **Dynamic Daily Volume**: Day-of-week seasonality (weekend shopping surges, weekday variations).
- **Rich Practical Schema**: Multi-item cart arrays, payment details, shipping carriers & tracking, promo codes, regional taxes.
- **Order Lifecycle CDC Transitions**: Status progressions (PENDING -> SHIPPED -> DELIVERED) testing Delta Lake MERGE.
- **Live Public API Support**: European Central Bank FX rates (Frankfurter) and live e-commerce carts (DummyJSON).
- **Partitioned Staging**: Writes NDJSON to `/Volumes/bronze_{env}/ecommerce/raw_landing/api_orders/year=YYYY/month=MM/day=DD/`.


In [0]:
%run /Repos/praveen6naik@gmail.com/Databricks-End-to-End-project/01_Config


In [0]:
# Task Widgets
dbutils.widgets.dropdown("target_env", "dev", ["dev", "prod", "qa"], "Target Environment")
dbutils.widgets.text("p_date", "yesterday", "Extraction Date (YYYY-MM-DD or yesterday)")
dbutils.widgets.text("p_api_url", "", "API Base URL (mock | live | dummyjson | frankfurter | custom)")
dbutils.widgets.text("p_api_key", "", "API Auth Key / Token")
dbutils.widgets.text("p_base_volume", "400", "Base Daily Volume")
dbutils.widgets.dropdown("p_include_lifecycle", "true", ["true", "false"], "Include Order Lifecycle CDC")
dbutils.widgets.dropdown("p_include_fx_rates", "true", ["true", "false"], "Fetch ECB Daily FX Rates")

p_env = dbutils.widgets.get("target_env").lower().strip()
p_date = dbutils.widgets.get("p_date").strip()
p_api_url = dbutils.widgets.get("p_api_url").strip()
p_api_key = dbutils.widgets.get("p_api_key").strip()
p_base_volume = int(dbutils.widgets.get("p_base_volume").strip() or "400")
p_lifecycle = dbutils.widgets.get("p_include_lifecycle").lower().strip() == "true"
p_fx_rates = dbutils.widgets.get("p_include_fx_rates").lower().strip() == "true"

print(f"[EXTRACT TASK] Env: '{p_env.upper()}', Target Date: '{p_date}', Mode: '{p_api_url or 'Enhanced Simulation'}', Base Vol: {p_base_volume}")


In [0]:
import sys
import os
import json
from datetime import datetime, timedelta, timezone

workspace_root = "/Workspace/Repos/praveen6naik@gmail.com/Databricks-End-to-End-project"
if workspace_root not in sys.path:
    sys.path.append(workspace_root)

from extract_api import (
    calculate_date_window,
    generate_mock_records,
    fetch_frankfurter_fx,
    fetch_dummyjson_orders,
    extract_from_api
)

start_dt, end_dt, date_str = calculate_date_window(p_date)
print(f"Extraction Window: {start_dt.isoformat()} <= created_at < {end_dt.isoformat()}")

records = []
api_mode = p_api_url.lower() if p_api_url else "mock"

if api_mode in ("dummyjson", "live"):
    print("Fetching from live DummyJSON public e-commerce API...")
    records = fetch_dummyjson_orders(start_dt, end_dt, limit=p_base_volume)
    if not records:
        print("Live API fallback: Generating rich synthetic transactions...")
        records = generate_mock_records(start_dt, end_dt, base_volume=p_base_volume, include_lifecycle_updates=p_lifecycle)
elif p_api_url and api_mode not in ("mock", "frankfurter"):
    print(f"Querying external REST API: {p_api_url}")
    try:
        records = extract_from_api(p_api_url, start_dt, end_dt, api_token=p_api_key or None)
    except Exception as exc:
        print(f"External API failed: {exc}. Falling back to rich simulation engine...")
        records = generate_mock_records(start_dt, end_dt, base_volume=p_base_volume, include_lifecycle_updates=p_lifecycle)
else:
    print("Generating realistic enterprise daily orders (day-of-week seasonality, multi-item carts, payments, shipping)...")
    records = generate_mock_records(start_dt, end_dt, base_volume=p_base_volume, include_lifecycle_updates=p_lifecycle)

print(f"✓ Total daily records prepared: {len(records):,}")


In [0]:
# Write Partitioned JSON to Unity Catalog Volume
landing_api_dir = f"{RAW_LANDING_VOLUME}/api_orders/year={start_dt.year:04d}/month={start_dt.month:02d}/day={start_dt.day:02d}"
file_name = f"orders_{date_str}.json"
full_path = f"{landing_api_dir}/{file_name}"

try:
    dbutils.fs.mkdirs(landing_api_dir)
except Exception:
    os.makedirs(landing_api_dir, exist_ok=True)

ndjson_content = "\n".join(json.dumps(r, default=str) for r in records) + "\n"

try:
    dbutils.fs.put(full_path, ndjson_content, overwrite=True)
    print(f"✓ Successfully staged {len(records)} JSON records to Volume via dbutils:\n  {full_path}")
except Exception:
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(ndjson_content)
    print(f"✓ Successfully wrote {len(records)} JSON records via OS filesystem:\n  {full_path}")

# Optional: Fetch and land daily FX rates
if p_fx_rates:
    fx_data = fetch_frankfurter_fx(date_str)
    if fx_data:
        fx_dir = f"{RAW_LANDING_VOLUME}/fx_rates/year={start_dt.year:04d}/month={start_dt.month:02d}/day={start_dt.day:02d}"
        fx_path = f"{fx_dir}/fx_rates_{date_str}.json"
        try:
            dbutils.fs.mkdirs(fx_dir)
            dbutils.fs.put(fx_path, json.dumps(fx_data, indent=2), overwrite=True)
        except Exception:
            os.makedirs(fx_dir, exist_ok=True)
            with open(fx_path, "w", encoding="utf-8") as f:
                json.dump(fx_data, f, indent=2)
        print(f"✓ Daily FX rates staged: {fx_path}")

dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "date": date_str, "records": len(records), "path": full_path}))
